# SBench KDD Apriori Execution

This notebook executes the data-mining and evaluation stages of the SBench KDD workflow. It consumes the existing `results.sqlite` database, groups `execution_items` into transactions, runs Apriori with `mlxtend`, generates association rules, and optionally exports the results.


## 1. Configuration

Tune these parameters for each KDD execution. Record the chosen values with the exported outputs.


In [ ]:
from __future__ import annotations

import json
import sqlite3
import warnings
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

MIN_SUPPORT = 0.10
MIN_CONFIDENCE = 0.50
MIN_LIFT = 1.00
MAX_ITEMSET_SIZE = 3

DROP_HIGH_SUPPORT_ITEMS = True
MAX_ITEM_SUPPORT = 0.95


In [ ]:
def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / 'sbench').is_dir() and (path / 'results.sqlite').exists():
            return path
    raise FileNotFoundError('Could not find repo root containing sbench/ and results.sqlite')


REPO_ROOT = find_repo_root(Path.cwd())
DATABASE_PATH = REPO_ROOT / 'results.sqlite'
DATABASE_URI = DATABASE_PATH.resolve().as_uri() + '?mode=ro'
EXECUTION_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT_DIR = REPO_ROOT / 'notebooks' / 'kdd_apriori_outputs' / EXECUTION_ID

REPO_ROOT, DATABASE_PATH, OUTPUT_DIR


## 2. Load Transactions

Each `execution_id` becomes one Apriori transaction. Each row in `execution_items` becomes one categorical item in that transaction.


In [ ]:
with sqlite3.connect(DATABASE_URI, uri=True) as connection:
    connection.row_factory = sqlite3.Row
    execution_rows = list(connection.execute('SELECT * FROM executions ORDER BY execution_id'))
    item_rows = list(connection.execute('SELECT execution_id, item FROM execution_items ORDER BY execution_id, item'))

transactions_by_execution: dict[str, set[str]] = {}
for row in item_rows:
    transactions_by_execution.setdefault(row['execution_id'], set()).add(row['item'])

transaction_ids = sorted(transactions_by_execution)
transactions = [sorted(transactions_by_execution[execution_id]) for execution_id in transaction_ids]
transaction_signatures = Counter(tuple(transaction) for transaction in transactions)

dataset_summary = {
    'execution_count': len(execution_rows),
    'transaction_count': len(transactions),
    'unique_transaction_itemsets': len(transaction_signatures),
    'transaction_item_count': len(item_rows),
    'harness_counts': dict(Counter(row['harness'] for row in execution_rows)),
    'track_counts': dict(Counter(row['track'] for row in execution_rows)),
    'status_counts': dict(Counter(row['status'] for row in execution_rows)),
    'deliverables_present_counts': dict(Counter(str(row['deliverables_present']) for row in execution_rows)),
    'parameters': {
        'algorithm': 'mlxtend.frequent_patterns.apriori',
        'transaction_encoder': 'mlxtend.preprocessing.TransactionEncoder',
        'min_support': MIN_SUPPORT,
        'min_confidence': MIN_CONFIDENCE,
        'min_lift': MIN_LIFT,
        'max_itemset_size': MAX_ITEMSET_SIZE,
        'drop_high_support_items': DROP_HIGH_SUPPORT_ITEMS,
        'max_item_support': MAX_ITEM_SUPPORT,
    },
}
dataset_summary


## 3. Encode And Diagnose Transactions

`mlxtend` expects a one-hot encoded table where each row is a transaction and each boolean column is an item. The item-support table highlights invariant items before Apriori runs.


In [ ]:
encoder = TransactionEncoder()
encoded_transactions = encoder.fit(transactions).transform(transactions) if transactions else []
transaction_frame = pd.DataFrame(encoded_transactions, columns=encoder.columns_)
transaction_frame.index = transaction_ids

item_support = pd.DataFrame({
    'item': transaction_frame.columns,
    'support_count': transaction_frame.sum(axis=0).astype(int).to_numpy(),
    'support': transaction_frame.mean(axis=0).to_numpy(),
}).sort_values(['support', 'item'], ascending=[False, True]).reset_index(drop=True)

high_support_items = item_support.loc[item_support['support'] >= MAX_ITEM_SUPPORT, 'item'].tolist()
if DROP_HIGH_SUPPORT_ITEMS:
    mining_frame = transaction_frame.drop(columns=high_support_items)
else:
    mining_frame = transaction_frame

dataset_summary['item_vocabulary_size'] = int(transaction_frame.shape[1])
dataset_summary['mining_item_vocabulary_size'] = int(mining_frame.shape[1])
dataset_summary['excluded_high_support_items'] = high_support_items if DROP_HIGH_SUPPORT_ITEMS else []

item_support.head(30)


## 4. Run Apriori

`mlxtend.frequent_patterns.apriori` returns frequent itemsets with support. By default this notebook mines `mining_frame`, which excludes items at or above `MAX_ITEM_SUPPORT` so global facts do not dominate the result table.


In [ ]:
def format_itemset(itemset: frozenset[str]) -> str:
    return ' | '.join(sorted(itemset))


frequent_itemsets = apriori(
    mining_frame,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_ITEMSET_SIZE,
) if not mining_frame.empty else pd.DataFrame(columns=['support', 'itemsets'])

if frequent_itemsets.empty:
    frequent_itemsets_export = pd.DataFrame(columns=['items', 'size', 'support_count', 'support'])
else:
    frequent_itemsets = frequent_itemsets.assign(
        size=frequent_itemsets['itemsets'].map(len),
        support_count=(frequent_itemsets['support'] * len(mining_frame)).round().astype(int),
        items=frequent_itemsets['itemsets'].map(format_itemset),
    ).sort_values(['support', 'size', 'items'], ascending=[False, True, True]).reset_index(drop=True)
    frequent_itemsets_export = frequent_itemsets[['items', 'size', 'support_count', 'support']]

frequent_itemsets_export.head(20)


## 5. Generate Association Rules

Rules are filtered by confidence and lift. The readable export keeps the core metrics needed for evaluation.


In [ ]:
rule_columns = ['antecedent', 'consequent', 'support_count', 'support', 'confidence', 'lift']

if frequent_itemsets.empty or not (frequent_itemsets['itemsets'].map(len) >= 2).any():
    rules_export = pd.DataFrame(columns=rule_columns)
else:
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=RuntimeWarning, module='mlxtend.frequent_patterns.association_rules')
        rules = association_rules(
            frequent_itemsets[['support', 'itemsets']],
            metric='confidence',
            min_threshold=MIN_CONFIDENCE,
        )
    rules = rules[rules['lift'] >= MIN_LIFT].copy()
    if rules.empty:
        rules_export = pd.DataFrame(columns=rule_columns)
    else:
        rules = rules.assign(
            antecedent=rules['antecedents'].map(format_itemset),
            consequent=rules['consequents'].map(format_itemset),
            support_count=(rules['support'] * len(mining_frame)).round().astype(int),
        ).sort_values(['lift', 'confidence', 'support'], ascending=[False, False, False]).reset_index(drop=True)
        rules_export = rules[rule_columns]

rules_export.head(20)


## 6. Export Results

Run this cell after reviewing the outputs. The exported files represent one KDD execution.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with (OUTPUT_DIR / 'dataset_summary.json').open('w', encoding='utf-8') as file:
    json.dump(dataset_summary, file, indent=2, sort_keys=True)

item_support.to_csv(OUTPUT_DIR / 'item_support.csv', index=False)
frequent_itemsets_export.to_csv(OUTPUT_DIR / 'frequent_itemsets.csv', index=False)
rules_export.to_csv(OUTPUT_DIR / 'association_rules.csv', index=False)

evaluation_template = '''# KDD Apriori Evaluation Notes

## Parameters

- algorithm: {algorithm}
- transaction_encoder: {transaction_encoder}
- min_support: {min_support}
- min_confidence: {min_confidence}
- min_lift: {min_lift}
- max_itemset_size: {max_itemset_size}
- drop_high_support_items: {drop_high_support_items}
- max_item_support: {max_item_support}

## Excluded High-Support Items

{excluded_items}

## Meaningful Rules

- TBD

## Trivial Or Misleading Rules

- TBD

## Dataset Limitations

- Current imported data may have limited variation in status and deliverable completion.
'''.format(
    excluded_items='\n'.join(f'- `{item}`' for item in dataset_summary['excluded_high_support_items']) or '- None',
    **dataset_summary['parameters'],
)

(OUTPUT_DIR / 'evaluation_notes.md').write_text(evaluation_template, encoding='utf-8')
OUTPUT_DIR
